## NOTEBOOK TIMER AND DATA DOWNLOAD

In [163]:
import time
import os
import json
import psutil

# Starting the notebook timer
notebook_start_time = time.time()
print("Notebook execution started...")

Notebook execution started...


We start the timer for the whole notebook in this cell, so when we press run all we can see the total time it took at the end.

In [164]:
!mysql -e "DROP DATABASE IF EXISTS cyclistic;"
!mysql -e "CREATE DATABASE cyclistic;"

First, we create our database called cyclistic. 

In [165]:
!mysql -e "SHOW DATABASES;"

+--------------------+
| Database           |
+--------------------+
| burke23201534      |
| cyclistic          |
| information_schema |
| mydb               |
| mysql              |
| newdb              |
| performance_schema |
| sys                |
+--------------------+


Lets take a look to make sure our database is there.

In [166]:
%%bash
CSV_NAME="trips_prototype.csv"
echo "Locating $CSV_NAME..."
if [ -f "$CSV_NAME" ]; then
    echo "File found! Inspecting data..."
    
    # Calculate human-readable file size and count the total rows
    FILE_SIZE=$(du -h "$CSV_NAME" | cut -f1)
    ROW_COUNT=$(wc -l < "$CSV_NAME")

    echo "-----------------------------------"
    echo "Data Inspection Summary:"
    echo "File size: $FILE_SIZE"
    echo "Total rows (including header): $ROW_COUNT"
    echo "-----------------------------------"
else
    echo "Error: $CSV_NAME could not be found. Make sure it is in the same folder as this notebook!"
fi

Locating trips_prototype.csv...
File found! Inspecting data...
-----------------------------------
Data Inspection Summary:
File size:  26M
Total rows (including header):   148046
-----------------------------------


Here we download part of our data for a prototype analysis.

In [167]:
%%bash
mysql -D "cyclistic" -e "
CREATE TABLE trips (
    ride_id VARCHAR(255) PRIMARY KEY,
    rideable_type VARCHAR(50),
    started_at DATETIME,
    ended_at DATETIME,
    start_station_name VARCHAR(255),
    start_station_id VARCHAR(255),
    end_station_name VARCHAR(255),
    end_station_id VARCHAR(255),
    start_lat DECIMAL(10, 8),
    start_lng DECIMAL(11, 8),
    end_lat DECIMAL(10, 8),
    end_lng DECIMAL(11, 8),
    member_casual VARCHAR(50)
);"
mysql -e "SET GLOBAL local_infile = 1;"

Now we create the table schema, we must make sure it matches the csv exactly. We also tell the MySQL server to globally allow local file uploads.

In [168]:
%%bash
CSV_FILE=$(ls *.csv | head -n 1)
echo "Loading $CSV_FILE into MySQL..."
time mysql --local-infile=1 -D "cyclistic" -e "
LOAD DATA LOCAL INFILE '$CSV_FILE' 
INTO TABLE trips 
FIELDS TERMINATED BY ',' 
ENCLOSED BY '\"' 
LINES TERMINATED BY '\n' 
IGNORE 1 ROWS;
"

Loading trips_prototype.csv into MySQL...



real	0m1.223s
user	0m0.010s
sys	0m0.040s


This is where we load in our data, and take note of how long it takes

In [169]:
%%bash
time mysql -E -D "cyclistic" -e "
SELECT 
    SUM(CASE WHEN ride_id IS NULL OR ride_id = '' THEN 1 ELSE 0 END) AS missing_ids,
    SUM(CASE WHEN start_station_name IS NULL OR start_station_name = '' THEN 1 ELSE 0 END) AS missing_start_stations,
    SUM(CASE WHEN end_station_name IS NULL OR end_station_name = '' THEN 1 ELSE 0 END) AS missing_end_stations,
    SUM(CASE WHEN end_lat IS NULL OR end_lng IS NULL THEN 1 ELSE 0 END) AS missing_end_coords,
    SUM(CASE WHEN member_casual IS NULL OR member_casual = '' THEN 1 ELSE 0 END) AS missing_user_types
FROM trips;
"

*************************** 1. row ***************************
           missing_ids: 0
missing_start_stations: 16102
  missing_end_stations: 17453
    missing_end_coords: 0
    missing_user_types: 0



real	0m0.095s
user	0m0.007s
sys	0m0.003s


Lets take a look at the data to ensure we have no missing data

## ANALYSIS

In [170]:
%%bash
time mysql -E -D "cyclistic" -e "
SELECT 
    REPLACE(REPLACE(member_casual, '\r', ''), '\n', '') AS user_type, 
    REPLACE(REPLACE(rideable_type, '\r', ''), '\n', '') AS bike_type, 
    COUNT(*) as total_rides
FROM trips
GROUP BY user_type, bike_type
ORDER BY user_type, total_rides DESC; 
"

*************************** 1. row ***************************
  user_type: casual
  bike_type: electric_bike
total_rides: 23996
*************************** 2. row ***************************
  user_type: casual
  bike_type: classic_bike
total_rides: 21765
*************************** 3. row ***************************
  user_type: casual
  bike_type: docked_bike
total_rides: 16284
*************************** 4. row ***************************
  user_type: member
  bike_type: classic_bike
total_rides: 37626
*************************** 5. row ***************************
  user_type: member
  bike_type: electric_bike
total_rides: 30180
*************************** 6. row ***************************
  user_type: member
  bike_type: docked_bike
total_rides: 18194



real	0m0.104s
user	0m0.011s
sys	0m0.004s


Here we can see the breakdown of total rides taken and type of bike used by customers with an anual membersihp (members) vs. customers who have purchased a single-ride/full-day pass (casual). 

In [171]:
%%bash
echo "--- Top 10 Start Stations ---"
time mysql -D "cyclistic" -e "SELECT start_station_name, COUNT(*) as count FROM trips WHERE start_station_name != '' GROUP BY start_station_name ORDER BY count DESC LIMIT 10;"

--- Top 10 Start Stations ---
start_station_name	count
Streeter Dr & Grand Ave	1916
Michigan Ave & Oak St	1151
Clark St & Elm St	1107
Wells St & Concord Ln	1063
Millennium Park	1047
Theater on the Lake	974
Wells St & Elm St	924
Indiana Ave & Roosevelt Rd	905
Clark St & Lincoln Ave	894
Kingsbury St & Kinzie St	870



real	0m0.084s
user	0m0.013s
sys	0m0.004s


Now we can see the top 10 start stations.

In [172]:
%%bash
echo "--- Top 10 End Stations ---"
time mysql -D "cyclistic" -e "SELECT end_station_name, COUNT(*) as count FROM trips WHERE end_station_name != '' GROUP BY end_station_name ORDER BY count DESC LIMIT 10;"

--- Top 10 End Stations ---
end_station_name	count
Streeter Dr & Grand Ave	1952
Michigan Ave & Oak St	1114
Wells St & Concord Ln	1097
Clark St & Elm St	1083
Millennium Park	1062
Theater on the Lake	1007
Wells St & Elm St	907
Broadway & Barry Ave	901
Clark St & Lincoln Ave	847
Clark St & Armitage Ave	824



real	0m0.078s
user	0m0.010s
sys	0m0.004s


Here we can see the 10 most popular end stations.

In [173]:
%%bash
echo "--- Average Ride Duration (Minutes) by User & Bike Type ---"
(time mysql -D "cyclistic" -e "
SELECT 
    REPLACE(REPLACE(member_casual, '\r', ''), '\n', '') AS user_type, 
    REPLACE(REPLACE(rideable_type, '\r', ''), '\n', '') AS bike_type, 
    ROUND(AVG(TIMESTAMPDIFF(MINUTE, started_at, ended_at)), 2) AS avg_duration_min
FROM trips
GROUP BY user_type, bike_type
ORDER BY user_type, avg_duration_min DESC;
") 2>&1

--- Average Ride Duration (Minutes) by User & Bike Type ---
user_type	bike_type	avg_duration_min
casual	docked_bike	57.18
casual	classic_bike	28.55
casual	electric_bike	17.55
member	classic_bike	13.47
member	docked_bike	12.05
member	electric_bike	11.65

real	0m0.127s
user	0m0.012s
sys	0m0.004s


This shows us the average duration of rides based on membership and bike type.

In [174]:
%%bash
echo " "
echo "--- Annual Members: Day x Hour Heatmap ---"
time mysql -D "cyclistic" -e "
SELECT 
    HOUR(started_at) AS hour_of_day,
    SUM(CASE WHEN DAYNAME(started_at) = 'Monday' THEN 1 ELSE 0 END) AS Monday,
    SUM(CASE WHEN DAYNAME(started_at) = 'Tuesday' THEN 1 ELSE 0 END) AS Tuesday,
    SUM(CASE WHEN DAYNAME(started_at) = 'Wednesday' THEN 1 ELSE 0 END) AS Wednesday,
    SUM(CASE WHEN DAYNAME(started_at) = 'Thursday' THEN 1 ELSE 0 END) AS Thursday,
    SUM(CASE WHEN DAYNAME(started_at) = 'Friday' THEN 1 ELSE 0 END) AS Friday,
    SUM(CASE WHEN DAYNAME(started_at) = 'Saturday' THEN 1 ELSE 0 END) AS Saturday,
    SUM(CASE WHEN DAYNAME(started_at) = 'Sunday' THEN 1 ELSE 0 END) AS Sunday
FROM trips
WHERE REPLACE(REPLACE(member_casual, '\r', ''), '\n', '') = 'member'
GROUP BY hour_of_day
ORDER BY hour_of_day;
"

 
--- Annual Members: Day x Hour Heatmap ---
hour_of_day	Monday	Tuesday	Wednesday	Thursday	Friday	Saturday	Sunday
0	66	51	58	99	104	202	203
1	36	27	30	44	57	137	173
2	33	15	17	20	28	94	88
3	23	15	8	17	24	44	50
4	26	31	28	27	37	26	36
5	129	165	155	148	145	45	43
6	363	475	452	432	365	139	113
7	729	865	907	804	641	243	179
8	827	950	992	911	690	410	259
9	530	517	571	581	497	626	434
10	432	404	473	419	434	732	644
11	511	518	497	522	560	880	795
12	630	674	607	677	707	933	879
13	572	659	648	646	700	896	858
14	601	619	623	648	743	949	867
15	734	780	747	758	863	945	864
16	1103	1214	1190	1141	1163	919	853
17	1395	1519	1578	1533	1270	877	787
18	1176	1278	1311	1227	1119	805	685
19	795	869	887	871	723	629	582
20	527	551	605	637	499	441	397
21	364	388	459	418	387	374	255
22	206	244	286	341	326	343	192
23	115	139	147	228	270	293	115



real	0m0.135s
user	0m0.011s
sys	0m0.004s


This cell generates a type of 'heatmap' that shows us the most popular times for users to take a ride. For example, in the annual members we see spikes on weekdays around 8a.m. and 5p.m., which strongly indicates a commuter pattern.

In [175]:
%%bash
echo "--- Casual Riders: Day x Hour Heatmap ---"
time mysql -D "cyclistic" -e "
SELECT 
    HOUR(started_at) AS hour_of_day,
    SUM(CASE WHEN DAYNAME(started_at) = 'Monday' THEN 1 ELSE 0 END) AS Monday,
    SUM(CASE WHEN DAYNAME(started_at) = 'Tuesday' THEN 1 ELSE 0 END) AS Tuesday,
    SUM(CASE WHEN DAYNAME(started_at) = 'Wednesday' THEN 1 ELSE 0 END) AS Wednesday,
    SUM(CASE WHEN DAYNAME(started_at) = 'Thursday' THEN 1 ELSE 0 END) AS Thursday,
    SUM(CASE WHEN DAYNAME(started_at) = 'Friday' THEN 1 ELSE 0 END) AS Friday,
    SUM(CASE WHEN DAYNAME(started_at) = 'Saturday' THEN 1 ELSE 0 END) AS Saturday,
    SUM(CASE WHEN DAYNAME(started_at) = 'Sunday' THEN 1 ELSE 0 END) AS Sunday
FROM trips
WHERE REPLACE(REPLACE(member_casual, '\r', ''), '\n', '') = 'casual'
GROUP BY hour_of_day
ORDER BY hour_of_day;
"


--- Casual Riders: Day x Hour Heatmap ---
hour_of_day	Monday	Tuesday	Wednesday	Thursday	Friday	Saturday	Sunday
0	114	63	73	91	160	328	334
1	60	57	34	55	94	283	265
2	38	27	27	31	31	154	178
3	29	21	18	16	29	78	80
4	22	16	16	16	14	30	52
5	36	43	38	39	39	39	47
6	98	101	127	113	96	64	64
7	166	200	195	200	184	116	116
8	207	254	229	259	225	283	189
9	209	220	208	242	235	416	357
10	281	229	218	248	302	640	551
11	367	297	307	297	388	892	790
12	436	357	355	350	548	1043	861
13	459	384	385	372	592	1172	979
14	508	372	385	425	588	1213	984
15	524	444	448	478	723	1208	974
16	606	537	576	648	750	1053	933
17	770	765	808	843	941	1035	831
18	668	712	738	818	861	920	692
19	478	559	547	583	656	738	560
20	346	353	431	434	461	564	378
21	265	294	354	363	403	517	360
22	205	244	277	341	362	523	271
23	126	152	166	233	363	481	190



real	0m0.115s
user	0m0.011s
sys	0m0.004s


This is the same type of heatmap for the causual riders.

In [176]:
%%bash
time mysql -E -D "cyclistic" -e "
SELECT 
    station_name, 
    SUM(activity_count) AS total_activity
FROM (
    SELECT start_station_name AS station_name, COUNT(*) AS activity_count
    FROM trips WHERE start_station_name != '' AND start_station_name IS NOT NULL
    GROUP BY start_station_name
    UNION ALL
    SELECT end_station_name AS station_name, COUNT(*) AS activity_count
    FROM trips WHERE end_station_name != '' AND end_station_name IS NOT NULL
    GROUP BY end_station_name
) AS combined_traffic
GROUP BY station_name
ORDER BY total_activity DESC
LIMIT 5;
"

*************************** 1. row ***************************
  station_name: Streeter Dr & Grand Ave
total_activity: 3868
*************************** 2. row ***************************
  station_name: Michigan Ave & Oak St
total_activity: 2265
*************************** 3. row ***************************
  station_name: Clark St & Elm St
total_activity: 2190
*************************** 4. row ***************************
  station_name: Wells St & Concord Ln
total_activity: 2160
*************************** 5. row ***************************
  station_name: Millennium Park
total_activity: 2109



real	0m0.153s
user	0m0.013s
sys	0m0.004s


This cell shows us which stations have most traffic (which stations have the most people starting + stopping at). This shows us which stations are most popular, and which would need the most upkeep.

In [177]:
%%bash
time mysql -D "cyclistic" -e "
SELECT 
    REPLACE(REPLACE(member_casual, '\r', ''), '\n', '') AS user_type, 
    DAYNAME(started_at) AS day_of_week, 
    COUNT(*) AS user_count
FROM trips
GROUP BY user_type, day_of_week
ORDER BY 
    user_type, 
    FIELD(day_of_week, 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday');
"

user_type	day_of_week	user_count
casual	Monday	7018
casual	Tuesday	6701
casual	Wednesday	6960
casual	Thursday	7495
casual	Friday	9045
casual	Saturday	13790
casual	Sunday	11036
member	Monday	11923
member	Tuesday	12967
member	Wednesday	13276
member	Thursday	13149
member	Friday	12352
member	Saturday	11982
member	Sunday	10351



real	0m0.153s
user	0m0.013s
sys	0m0.007s


Here we can see whch days of the week are most popular for causal and member riders.

## TOTAL TIME AND MEMORY

In the last cell below, we calculate the total time and memory this notebook took to process our prototype data with sql.

In [178]:
# Calculate execution time
notebook_end_time = time.time()
total_seconds = notebook_end_time - notebook_start_time
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)

print("-" * 40)
print(f"Total SQL Notebook Execution Time: {minutes} minutes and {seconds} seconds.")

process = psutil.Process(os.getpid())
memory_info = process.memory_info()
memory_usage_mb = memory_info.rss / (1024 * 1024)

print(f"Memory Usage: {memory_usage_mb:.2f} MB")
print("-" * 40)

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
results_dir = os.path.join(project_root, "results")
os.makedirs(results_dir, exist_ok=True)
metrics_file_path = os.path.join(results_dir, "metrics.json")

if os.path.exists(metrics_file_path):
    with open(metrics_file_path, "r") as f:
        try:
            metrics_data = json.load(f)
        except json.JSONDecodeError:
            metrics_data = {} 
else:
    metrics_data = {}

metrics_data["sql_analysis"] = {
    "total_execution_time_seconds": round(total_seconds, 2),
    "total_execution_time_formatted": f"{minutes}m {seconds}s",
    "memory_usage_mb": round(memory_usage_mb, 2)
}

with open(metrics_file_path, "w") as f:
    json.dump(metrics_data, f, indent=4)

print(f"Metrics successfully updated in: {metrics_file_path}")

----------------------------------------
Total SQL Notebook Execution Time: 0 minutes and 3 seconds.
Memory Usage: 36.56 MB
----------------------------------------
Metrics successfully updated in: /Users/josieburke/Desktop/Proj/Cyclistic_Data/results/metrics.json
